In [ ]:
!python 01_data_collection.py        

Built a grid of 16 candidate sites.
[1/16] fetching 22.5, 70.5 ...
[2/16] fetching 22.5, 71.8333 ...
[3/16] fetching 22.5, 73.1667 ...
[4/16] fetching 22.5, 74.5 ...
[5/16] fetching 23.8333, 70.5 ...
[6/16] fetching 23.8333, 71.8333 ...
[7/16] fetching 23.8333, 73.1667 ...
[8/16] fetching 23.8333, 74.5 ...
[9/16] fetching 25.1667, 70.5 ...
[10/16] fetching 25.1667, 71.8333 ...
[11/16] fetching 25.1667, 73.1667 ...
[12/16] fetching 25.1667, 74.5 ...
[13/16] fetching 26.5, 70.5 ...
[14/16] fetching 26.5, 71.8333 ...


In [2]:
import pandas as pd

csv = pd.read_csv("raw_sites_combined.csv")
csv

,date,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,ALLSKY_SFC_SW_DNI,ALLSKY_SFC_SW_DIFF,ALLSKY_KT,T2M,QV2M,RH2M,WS10M,PRECTOTCORR,lat,lon
0,2026-01-01,4.1515,4.3738,4.6284,1.5425,0.61,20.11,8.80,63.07,3.16,0.53,22.5,70.5
1,2026-01-02,4.5000,4.5108,5.7041,1.3294,0.66,18.37,6.51,53.82,4.00,0.00,22.5,70.5
2,2026-01-03,4.3838,4.3889,5.1293,1.4633,0.64,17.20,5.63,50.69,3.41,0.00,22.5,70.5
3,2026-01-04,4.6224,4.6327,6.1514,1.2420,0.68,17.65,5.17,46.12,3.45,0.00,22.5,70.5
4,2026-01-05,3.9922,4.0003,3.9893,1.6202,0.58,18.61,5.63,46.73,3.15,0.00,22.5,70.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4715,2026-10-18,4.2372,-999.0000,-999.0000,-999.0000,-999.00,27.72,12.02,52.86,3.37,1.29,26.5,74.5
4716,2026-10-19,4.7232,-999.0000,-999.0000,-999.0000,-999.00,27.31,11.04,50.44,2.11,0.05,26.5,74.5
4717,2026-10-20,4.7546,-999.0000,-999.0000,-999.0000,-999.00,27.72,10.07,45.12,1.78,0.00,26.5,74.5
4718,2026-10-21,4.9015,-999.0000,-999.0000,-999.0000,-999.00,27.55,9.95,44.86,2.33,0.00,26.5,74.5


In [3]:
!python 02_feature_engineering.py    

/Users/gera/Desktop/DSF-Project/files/02_feature_engineering.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["lat", "lon"], group_keys=False).apply(clean_site)
Cleaning done. Nulls remaining: 0
/Users/gera/Desktop/DSF-Project/files/02_feature_engineering.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["lat", "lon"], group_keys=False).apply(add_lag_features)
Leakage check pa

In [4]:
!python 03_train_forecast_model.py 

Train: 3680 rows up to 2026-08-25, Test: 912 rows after.
Persistence baseline       MAE=0.0550  RMSE=0.0939
Climatology baseline       MAE=0.0707  RMSE=0.1084
Random Forest (main model)  MAE=0.1434  RMSE=0.1588
R2: -1.2983

SKILL SCORE vs persistence: -69.2% (positive = model beats the naive guess)
80% interval empirical coverage: 62.3% (target: ~80%)

Saved forecast_model.joblib, forecast_lower.joblib, forecast_upper.joblib


In [5]:
!python 04_site_metrics.py         

/Users/gera/Desktop/DSF-Project/files/04_site_metrics.py:50: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  site_table = df.groupby(["lat", "lon"]).apply(site_summary).reset_index()
Saved site_metrics_table.csv with 16 sites.
    lat     lon  mean_daily_yield_kwh  variability_cv  seasonal_stability  regime_label
22.5000 70.5000            925.335207        0.262818            0.490052     Low Yield
22.5000 71.8333            921.367756        0.271403            0.502905     Low Yield
22.5000 73.1667            914.049037        0.274977            0.512606     Low Yield
22.5000 74.5000            896.620097        0.293286            0.472797     Low Yield
23.8333 70.5000            928.681294        

In [6]:
!python 05_ahp_scoring.py          

AHP weights: {'mean_daily_yield_kwh': np.float64(0.6), 'variability_cv': np.float64(0.2), 'seasonal_stability': np.float64(0.2)}
Consistency Ratio: 0.000 (OK, below 0.1)

Saved site_ranking_final.csv. Top 5 sites:
    lat     lon  suitability_score  top_k_freq  regime_label
26.5000 70.5000          97.030850         1.0 High & Stable
26.5000 71.8333          91.419150         1.0 High & Stable
25.1667 70.5000          86.380833         1.0 High & Stable
25.1667 71.8333          84.648808         1.0 High & Stable
26.5000 73.1667          70.175488         1.0 High & Stable


In [8]:
!python 05_ahp_scoring.py          


AHP weights: {'mean_daily_yield_kwh': np.float64(0.6), 'variability_cv': np.float64(0.2), 'seasonal_stability': np.float64(0.2)}
Consistency Ratio: 0.000 (OK, below 0.1)

Saved site_ranking_final.csv. Top 5 sites:
    lat     lon  suitability_score  top_k_freq  regime_label
26.5000 70.5000          97.030850         1.0 High & Stable
26.5000 71.8333          91.419150         1.0 High & Stable
25.1667 70.5000          86.380833         1.0 High & Stable
25.1667 71.8333          84.648808         1.0 High & Stable
26.5000 73.1667          70.175488         1.0 High & Stable
Saved ahp_weights.json and reference_stats.json (copy both into api/ alongside sites_precomputed.json)


In [9]:
!python 06_precompute_all.py

Saved sites_precomputed.json with 16 sites.
Copy this file (and the three .joblib files) into api/ before running the backend.
